<h2 style="color:red; text-align:left; font-size:50px; font-weight:bold; text-decoration:underline;">
Cal Fire
</h2>

## Appendix B: *WildFires*
- *Version Number: 3.0*
---
### Contents  
> 1. *Wildfire Count Data* 
> 2. *Wildfire Damage Data*
> 3. *Combine Datasets*
> 4. *Export File*
---
### Notes
---
### Inputs
> - **CAL FIRE Damage Inspection (DINS)** Data: <https://data.ca.gov/dataset/cal-fire-damage-inspection-dins-data>
>   - raw dataset = '.../data/raw/POSTFIRE_MASTER_DATA_SHARE_2064760709534146017.csv'
> - **Calfire Incident** Data: <https://www.fire.ca.gov/incidents>
>   - raw dataset = '.../data/raw/mapdatall.csv'
---
### Outputs  
- `fire_data.csv.csv` - cleaned dataset of fire related variables 
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# utility function that prints details of a datset
from src.data_utils import basic_explore

# utility function printing relevant details to check the health after a dataset merge
from src.data_utils import post_merge_check

# Function to map damages
from src.plot_utils import plot_fire_damage

### Third Party Dependencies  
---

In [2]:
# core data handling
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

## 1. Total Fire Count

### Load `CALfires.csv` - Wildfire Incident Data
Data obtained from [CAL FIRE Incident Data](https://www.fire.ca.gov/incidents)\
Dates of fire damage information: 01/01/2018 - 01/23/2025

> Variables Used:
> - `Fire name`
> - `Date started` and `Date Out` - Start and extinquishing date and time of wildfire incident 
> - `Total_Cost_Estimated` - in dollar amount
> - `County`, `Latitude` and `Longitude` - Spatial factors

Workflow:
- filter dataset by wildfire type
- column formatting
- fill NA values
- Ensure spatial accuracy of coordinates
- Format date and time columns

In [3]:
# Load data
allfires = pd.read_csv("../data/raw/Calfires.csv",low_memory = False)
allfires

,Unnamed: 0,SourceOID,ContainmentDateTime,IncidentSize,EstimatedCostToDate,FinalAcres,FireBehaviorGeneral,FireBehaviorGeneral1,FireBehaviorGeneral2,FireBehaviorGeneral3,...,LocalIncidentIdentifier,POOCity,POOCounty,POOState,PredominantFuelGroup,PredominantFuelModel,PrimaryFuelModel,SecondaryFuelModel,UniqueFireIdentifier,EstimatedFinalCost
0,0,7747595,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,066100,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2020-CALAC-066100,NaN
1,1,6384391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,009269,NaN,San Diego,US-CA,NaN,NaN,NaN,NaN,2019-CAMVU-009269,NaN
2,3,22499589,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,163915,NaN,Riverside,US-CA,NaN,NaN,NaN,NaN,2021-CARRU-163915,NaN
3,4,23869477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,396331,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2022-CALAC-396331,NaN
4,17,23247195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,008436,NaN,San Luis Obispo,US-CA,NaN,NaN,NaN,NaN,2020-CASLU-008436,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84534,364796,38438381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,242944,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2025-CALAC-242944,NaN
84535,364802,38438397,7/13/2025 4:02:20 PM,1.7,NaN,NaN,NaN,NaN,NaN,NaN,...,020730,NaN,Sacramento,US-CA,NaN,NaN,NaN,NaN,2025-CAAEU-020730,NaN
84536,364803,38438398,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,...,000813,NaN,Glenn,US-CA,NaN,NaN,NaN,NaN,2025-CAMNF-000813,NaN
84537,364819,38438435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,243075,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2025-CALAC-243075,NaN


In [4]:
# Filter for wildfire incidents only
wildfires = allfires[allfires['IncidentTypeCategory']=='WF']

# rename for convenience
wildfires = wildfires.rename(columns={
    'InitialLatitude': 'Fire_Latitude',
    'InitialLongitude': 'Fire_Longitude',
    'FireDiscoveryDateTime':'Date',
    'POOCounty':'County'
})

# Drop entries with crucial null variables, may be manually fixed soon
wildfires = wildfires.dropna(subset=['Fire_Latitude', 'Fire_Longitude','Date'])


# Filter Coordinates
## Keep only coordinates that fall inside California boundaries since
## some entries have incorrect latitude and longitude values. 
## To be manually corrected in the future but are filtered out for now.

wildfires = wildfires[
    (wildfires['Fire_Latitude'] >= 32.5) & 
    (wildfires['Fire_Latitude'] <= 42) &
    (wildfires['Fire_Longitude'] >= -124.5) &
    (wildfires['Fire_Longitude'] <= -114)
]

# Format date and time columns
wildfires['Date'] = pd.to_datetime(wildfires['Date']).dt.date

# Fill Remaining NA values
## Entries with no cost present are assumed to have a damage cost 0 dollars.
wildfires['EstimatedFinalCost'] = wildfires['EstimatedFinalCost'].fillna(0)

drop =['Unnamed: 0', 'SourceOID', 'ContainmentDateTime', 'IncidentSize','FireBehaviorGeneral','EstimatedCostToDate','FireBehaviorGeneral1', 'FireBehaviorGeneral2', 'FireBehaviorGeneral3',
 'FireCause', 'FireCauseGeneral', 'FireCauseSpecific','IncidentShortDescription', 'IncidentTypeKind',
  'IrwinID','LocalIncidentIdentifier', 'POOCity', 'POOState','PredominantFuelGroup', 'PredominantFuelModel', 
  'PrimaryFuelModel','SecondaryFuelModel', 'UniqueFireIdentifier','IncidentTypeCategory','FireOutDateTime']

all_fires = wildfires.drop(columns=drop)

## Change Fire Name Field
all_fires['Fire Name'] = all_fires['IncidentName'].str.lower()
all_fires = all_fires.drop(columns='IncidentName')

In [5]:
all_fires = all_fires.fillna(0)

## 2. Fire Damage Processing

### 2.1 Load File
`Damage_data_master` - Wildfire Incident Data on Structural Damage

Data obtained from CALFIRE website. Includes data on the degree of damages for every location since 1950. 

- Fire name and date started
- Damage levels (structures destroyed/damaged)
- County, Latitude, Longitude of event

In [6]:
raw_damages = pd.read_csv("../data/raw/Damage_data_master.csv",low_memory = False)
basic_explore(raw_damages)

Rows:  130722  Columns:  46
Duplicates  0
Total NA values:  1381634  of  6013212 datapoints


In [7]:
raw_damages.columns

Index(['OBJECTID', '* Damage', '* Street Number', '* Street Name',
       '* Street Type (e.g. road, drive, lane, etc.)',
       'Street Suffix (e.g. apt. 23, blding C)', '* City', 'State', 'Zip Code',
       '* CAL FIRE Unit', 'County', 'Community', 'Battalion',
       '* Incident Name', 'Incident Number (e.g. CAAEU 123456)',
       'Incident Start Date', 'Hazard Type',
       'If Affected 1-9% - Where did fire start?',
       'If Affected 1-9% - What started fire?',
       'Structure Defense Actions Taken', '* Structure Type',
       'Structure Category', '# Units in Structure (if multi unit)',
       '# of Damaged Outbuildings < 120 SQFT',
       '# of Non Damaged Outbuildings < 120 SQFT', '* Roof Construction',
       '* Eaves', '* Vent Screen', '* Exterior Siding', '* Window Pane',
       '* Deck/Porch On Grade', '* Deck/Porch Elevated',
       '* Patio Cover/Carport Attached to Structure',
       '* Fence Attached to Structure', 'Distance - Propane Tank to Structure',
       'Dis

### 2.2 Preprocess Fire Damage Records

- Filtered down to essential columns: fire name, type, category, damage, date, and county.
- Renamed columns for consistency.
- Retained only fire-related records (excluding other hazard types).
- Convert the start date to `datetime.date` and rename.

In [8]:
# filter to columns to keep
old_col_names = ['* Damage','County','Incident Start Date',
                 'Hazard Type', 'Structure Category','* Structure Type','* Incident Name',
                'Latitude','Longitude']

damages = raw_damages[old_col_names]

# rename for convenience
damages.columns = ['Damage','County', 'Start','Hazard','Category',
                  'Type','Fire Name','Lat','Lon']

# Filter for only fires, drop Hazard category, it is no longer needed
damages = damages[damages['Hazard'] == 'Fire']
damages = damages.drop(columns = ['Hazard'])

# filter the dataset to current operating time range
damages['Date'] = pd.to_datetime(damages['Start'], format='mixed').dt.date
damages = damages.drop(columns = ['Start'])

In [9]:
basic_explore(damages)

Rows:  130720  Columns:  8
Duplicates  108
Total NA values:  30  of  1045760 datapoints


In [10]:
damages.isna().sum()

Damage        0
County       30
Category      0
Type          0
Fire Name     0
Lat           0
Lon           0
Date          0
dtype: int64

**Note:**  
Approximately 30 records were missing county names in the original data. However, ZIP code and address information allowed inference of the correct county.  
Due to the small number, these corrections were applied manually by index.  
- Rows `78763–78773`: Riverside County  
- Rows `78435–78451`: Yuba County  
- Rows `88665` and `88681`: Tulare County  

These updates ensure completeness for spatial grouping and aggregation in downstream analysis.

In [11]:
riverside_ids = range(78763,78774)
yuba_ids = range(78435,78452)
damages.loc[riverside_ids,'County'] = 'Riverside'
damages.loc[yuba_ids,'County'] = 'Yuba'
damages.loc[88665,'County'] = 'Tulare'
damages.loc[88681,'County'] = 'Tulare'

In [12]:
damages.head(2)

,Damage,County,Category,Type,Fire Name,Lat,Lon,Date
0,No Damage,Solano,Single Residence,Single Family Residence Multi Story,Quail,38.474960,-122.044465,2020-06-06
1,Affected (1-9%),Solano,Single Residence,Single Family Residence Single Story,Quail,38.477442,-122.043252,2020-06-06


### 2.3 Calculate Damages

#### Estimate Dollar Value of Structure Damage

To approximate the economic cost of fire damage, a base dollar value was assigned to each general structure type:

In [13]:
base_value_map = {
    'Single Residence': 800000,
    'Multiple Residence': 1000000,
    'Mixed Commercial/Residential': 1500000,
    'Nonresidential Commercial': 2000000,
    'Other Minor Structure': 10000,
    'Infrastructure': 1000000,
    'Agriculture': 10000,
}

#### Value Modifiers for Building Descriptors

The estimated dollar value for each damaged structure is adjusted based on detailed descriptors. These modifiers scale the base value assigned to the general structure type.

> **Note:** These modifiers are estimates that reflect relative structural value and are not based on official costs.

In [14]:
type_modifier_map = {
    'Single Family Residence Multi Story': 1.0,
    'Single Family Residence Single Story': 0.95,
    'Utility Misc Structure': 0.1,
    'Mobile Home Double Wide': 0.5,
    'Motor Home': 0.3,
    'Multi Family Residence Multi Story': 1.2,
    'Commercial Building Single Story': 1.0,
    'Mobile Home Single Wide': 0.4,
    'Mixed Commercial/Residential': 1.5,
    'Mobile Home Triple Wide': 0.6,
    'Infrastructure': 0.8,
    'School': 1.3,
    'Multi Family Residence Single Story': 1.1,
    'Commercial Building Multi Story': 1.4,
    'Church': 1.0,
    'Hospital': 1.6,
    'Agriculture': 0.2,
    'Single Famliy Residence Single Story': 0.95,
    'Utility or Miscellaneous Structure > 120 sqft':0.1
}

#### Lookup Table: Estimated Damage Values by Structure Type and Descriptor

The estimated monetary damage per structure is computed by multiplying:
- A base value associated with the general **structure category** (Single Residence)
- A modifier associated with the **building descriptor** (Single Story, Mobile Home)

This matrix provides a quick reference for assigning dollar estimates to individual records based on their structure type and category.

> **Formula:**  
> `Estimated Value = Base Value × Type Modifier`

In [15]:
# Create a DataFrame from the product of categories and types
value_matrix = pd.DataFrame(
    [(cat, typ, base_value_map[cat] * type_modifier_map[typ]) for cat in base_value_map for typ in type_modifier_map],
    columns=['Category', 'Type', 'Adjusted Value']
)

# Pivot the DataFrame to get a matrix format
value_matrix_df = value_matrix.pivot(index='Category', columns='Type', values='Adjusted Value')

# Display the resulting matrix
value_matrix_df

Type,Agriculture,Church,Commercial Building Multi Story,Commercial Building Single Story,Hospital,Infrastructure,Mixed Commercial/Residential,Mobile Home Double Wide,Mobile Home Single Wide,Mobile Home Triple Wide,Motor Home,Multi Family Residence Multi Story,Multi Family Residence Single Story,School,Single Family Residence Multi Story,Single Family Residence Single Story,Single Famliy Residence Single Story,Utility Misc Structure,Utility or Miscellaneous Structure > 120 sqft
Category,,,,,,,,,,,,,,,,,,,
Agriculture,2000.0,10000.0,14000.0,10000.0,16000.0,8000.0,15000.0,5000.0,4000.0,6000.0,3000.0,12000.0,11000.0,13000.0,10000.0,9500.0,9500.0,1000.0,1000.0
Infrastructure,200000.0,1000000.0,1400000.0,1000000.0,1600000.0,800000.0,1500000.0,500000.0,400000.0,600000.0,300000.0,1200000.0,1100000.0,1300000.0,1000000.0,950000.0,950000.0,100000.0,100000.0
Mixed Commercial/Residential,300000.0,1500000.0,2100000.0,1500000.0,2400000.0,1200000.0,2250000.0,750000.0,600000.0,900000.0,450000.0,1800000.0,1650000.0,1950000.0,1500000.0,1425000.0,1425000.0,150000.0,150000.0
Multiple Residence,200000.0,1000000.0,1400000.0,1000000.0,1600000.0,800000.0,1500000.0,500000.0,400000.0,600000.0,300000.0,1200000.0,1100000.0,1300000.0,1000000.0,950000.0,950000.0,100000.0,100000.0
Nonresidential Commercial,400000.0,2000000.0,2800000.0,2000000.0,3200000.0,1600000.0,3000000.0,1000000.0,800000.0,1200000.0,600000.0,2400000.0,2200000.0,2600000.0,2000000.0,1900000.0,1900000.0,200000.0,200000.0
Other Minor Structure,2000.0,10000.0,14000.0,10000.0,16000.0,8000.0,15000.0,5000.0,4000.0,6000.0,3000.0,12000.0,11000.0,13000.0,10000.0,9500.0,9500.0,1000.0,1000.0
Single Residence,160000.0,800000.0,1120000.0,800000.0,1280000.0,640000.0,1200000.0,400000.0,320000.0,480000.0,240000.0,960000.0,880000.0,1040000.0,800000.0,760000.0,760000.0,80000.0,80000.0


#### Assign Estimated Dollar Values to Each Record

Using the lookup matrix, each damage record is assigned an estimated economic value by matching its structure `Category` and `Type`.

- The assigned value represents an approximate replacement or reconstruction cost.
- These values are based on estimates and are used to generate a composite damage index for modeling purposes.

Validation:
- Any Rows with unmatched (Category, Type) combinations are flagged (`NaN`).


In [16]:
damages['Adjusted Value'] = damages.apply(
    lambda row: value_matrix_df.loc[row['Category'], row['Type']], axis=1
)

print('Adjusted Values NA values : ', damages['Adjusted Value'].isna().sum())
total = damages['Adjusted Value'].values.sum()
print(f"Total Adjusted Value: ${total:,.0f}")

Adjusted Values NA values :  0
Total Adjusted Value: $77,939,291,500


In [17]:
damage_weights = {
    'Destroyed (>50%)': 1.0,
    'Major (26-50%)': 0.4,
    'Minor (10-25%)': 0.2,
    'Affected (1-9%)': 0.05,
    'Inaccessible': 0.3,
    'No Damage': 0.0
}

In [18]:
# Assume each row is one structure. Map category to loss fraction:
damages['Loss Fraction'] = damages['Damage'].map(damage_weights)

# Calculate estimated damage per structure
damages['Estimated Damage'] = damages['Loss Fraction'] * damages['Adjusted Value']

#### Sanity Checks

In [19]:
damages.isna().sum()

Damage              0
County              0
Category            0
Type                0
Fire Name           0
Lat                 0
Lon                 0
Date                0
Adjusted Value      0
Loss Fraction       0
Estimated Damage    0
dtype: int64

In [20]:
damages['Damage'].value_counts()

Damage
Destroyed (>50%)    70127
No Damage           53051
Affected (1-9%)      5023
Minor (10-25%)       1337
Major (26-50%)        706
Inaccessible          476
Name: count, dtype: int64

#### Aggregate Damage Estimates by Fire Event

Now that each damage record has an estimated dollar value, we summarize the data:

- Group by **Fire Name** and **Fire Start Date**.
- Sum the `Adjusted Value` to get total `Estimated Damage` per fire event.

In [21]:
fire_damages = damages.groupby(['Fire Name', 'Date']).agg({
    'Estimated Damage': 'sum',
    'Lat': 'first',
    'Lon': 'first',
    'County': 'first'
}).reset_index()

In [34]:
fire_damages.rename(columns={'Lat':'Fire_Latitude','Lon':'Fire_Longitude'}, inplace = True)

## 3. Combine Datasets

In [37]:
fire_damages.head(2)

,Fire Name,Date,Estimated Damage,Fire_Latitude,Fire_Longitude,County
0,46th,2019-10-31,6834400.0,33.985706,-117.416998,Riverside
1,Aborn,2019-07-15,760000.0,37.320320,-121.754095,Santa Clara


In [38]:
all_fires.head(2)

,FinalAcres,Date,Fire_Latitude,Fire_Longitude,County,EstimatedFinalCost,Fire Name
0,0.0,2020-02-28,33.808980,-118.18070,Los Angeles,0.0,lac-066100
2,0.0,2021-11-25,33.782437,-117.22858,Riverside,0.0,e 4th st /s d st


In [63]:
fire_data = pd.concat([all_fires, fire_damages], ignore_index=True, sort=False)

In [64]:
fire_data.sort_values(by='Date', inplace=True)

In [65]:
fire_data['Estimated Damage'] = fire_data['Estimated Damage'].fillna(fire_data['EstimatedFinalCost'])
fire_data = fire_data.drop(columns='EstimatedFinalCost')

In [66]:
basic_explore(fire_data)

Rows:  77103  Columns:  7
Duplicates  111
Total NA values:  312  of  539721 datapoints


In [67]:
fire_data = fire_data.drop_duplicates()

In [68]:
basic_explore(fire_data)

Rows:  76992  Columns:  7
Duplicates  0
Total NA values:  312  of  538944 datapoints


In [69]:
fire_data.isna().sum()

FinalAcres          312
Date                  0
Fire_Latitude         0
Fire_Longitude        0
County                0
Fire Name             0
Estimated Damage      0
dtype: int64

In [70]:
fire_data = fire_data.fillna(0)

In [71]:
# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2025-01-23').date()

In [72]:
fire_data['Date'] = pd.to_datetime(fire_data['Date']).dt.date
fire_data = fire_data[fire_data['Date']>= FIRST_DATE]
fire_data = fire_data[fire_data['Date']<= LAST_DATE]

In [73]:
fire_data

,FinalAcres,Date,Fire_Latitude,Fire_Longitude,County,Fire Name,Estimated Damage
40870,0.0,2018-01-01,37.802100,-118.469700,Mono,yellow jacket,0.0
25936,0.0,2018-01-01,34.031731,-117.950401,Los Angeles,lac-393275,0.0
8099,0.0,2018-01-01,34.682388,-118.085892,Los Angeles,lac-393502,0.0
24749,0.0,2018-01-01,33.996208,-117.870201,Los Angeles,lac-000253,0.0
1956,0.0,2018-01-01,34.674992,-118.132889,Los Angeles,lac-393386,0.0
...,...,...,...,...,...,...,...
70736,0.0,2025-01-23,34.674870,-118.160690,Los Angeles,lac-030571,0.0
70737,0.0,2025-01-23,33.923190,-118.209760,Los Angeles,lac-030642,0.0
70738,0.0,2025-01-23,34.031560,-118.052350,Los Angeles,lac-030649,0.0
70729,0.0,2025-01-23,34.074640,-117.750170,Los Angeles,lac-030376,0.0


## 4. Export File

In [74]:
fire_data.to_csv("../data/processed/fire_data.csv",index=False)
print("All datasets saved successfully.")

All datasets saved successfully.
